# Rubik's Cube Optimization Sandbox

This notebook is deliberately **one step before diffusion**.

It does not use Stable Diffusion or any text prompt yet.
Instead, it teaches the smallest possible optimization loop for this project:

`six source face images -> Rubik's arrangement operator -> solved/scrambled outputs -> loss -> optimization`

If this notebook makes sense and behaves the way we expect, then the diffusion step later becomes much less mysterious.


## What This Notebook Teaches

By the end of this notebook we will have proved five things:

1. the public repo can bootstrap inside Colab
2. a **differentiable** torch version of the Rubik's operator matches the repo's PIL renderer
3. we can define tiny toy targets for one solved face and one scrambled face
4. we can optimize the six source faces directly as pixels
5. we can save the learned source faces and rendered outputs back into the repo folder

Run the cells top to bottom. Keep the image size small on purpose. The goal here is understanding, not beauty.


In [ ]:
import importlib
import os
import platform
import subprocess
import sys
from pathlib import Path

for module_name, pip_name in [('PIL', 'Pillow'), ('matplotlib', 'matplotlib')]:
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name])

try:
    import torch
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'Torch is expected to already be available in Colab. '
        'If this fails, restart into a standard Colab Python runtime and run again.'
    ) from exc

import matplotlib.pyplot as plt

print('Python:', sys.version)
print('Platform:', platform.platform())
print('In Colab runtime:', 'COLAB_RELEASE_TAG' in os.environ)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/netalondon/rubiks-diffusion-illusion.git'
REPO_DIR = Path('/content/rubiks-diffusion-illusion')

if REPO_DIR.exists():
    print(f'Reusing existing repo at {REPO_DIR}')
else:
    subprocess.check_call(['git', 'clone', REPO_URL, str(REPO_DIR)])

requirements_path = REPO_DIR / 'requirements.txt'
if requirements_path.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)])

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo ready:', REPO_DIR)


In [ ]:
import json

import numpy as np
import torch
from PIL import Image
from python_bridge.rubiks_illusion_operator import render_all_arrangements, save_rendered_faces

SPEC_PATH = REPO_DIR / 'public' / 'generated' / 'rubiks-illusion-spec.json'
spec = json.loads(SPEC_PATH.read_text())
prime_images = spec['primeImages']
source_grid_size = spec['sourceGridSize']
face_to_index = {face_id: index for index, face_id in enumerate(prime_images)}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Prime images:', prime_images)
print('Arrangement names:', list(spec['arrangements'].keys()))
print('Grid size:', source_grid_size)
print('Working device:', device)


In [ ]:
import matplotlib.pyplot as plt

PALETTE = torch.tensor(
    [
        [0.86, 0.23, 0.21],
        [0.17, 0.61, 0.33],
        [0.18, 0.42, 0.87],
        [0.90, 0.68, 0.19],
        [0.67, 0.31, 0.83],
        [0.18, 0.74, 0.78],
    ],
    dtype=torch.float32,
)


def torch_image_to_numpy(image):
    if isinstance(image, Image.Image):
        return np.asarray(image.convert('RGB')).astype(np.float32) / 255.0

    if isinstance(image, torch.Tensor):
        tensor = image.detach().cpu().clamp(0.0, 1.0)
        return tensor.permute(1, 2, 0).numpy()

    raise TypeError(f'Unsupported image type: {type(image)!r}')


def tensor_to_pil(image):
    array = (image.detach().cpu().clamp(0.0, 1.0).permute(1, 2, 0).numpy() * 255.0).round().astype(np.uint8)
    return Image.fromarray(array)


def pil_to_tensor(image, device=None):
    array = np.asarray(image.convert('RGB')).astype(np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1)
    if device is not None:
        tensor = tensor.to(device)
    return tensor


def batch_to_pil_face_dict(batch, face_order):
    return {face_id: tensor_to_pil(batch[index]) for index, face_id in enumerate(face_order)}


def clone_rendered_to_cpu(rendered):
    return {
        arrangement_name: {
            face_id: image.detach().cpu().clone()
            for face_id, image in faces.items()
        }
        for arrangement_name, faces in rendered.items()
    }


def show_face_dict(face_dict, title, face_order):
    cols = min(3, max(1, len(face_order)))
    rows = (len(face_order) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.2 * rows))
    axes = np.asarray(axes, dtype=object).reshape(rows, cols)
    fig.suptitle(title, fontsize=14)

    for axis in axes.flat:
        axis.axis('off')

    for index, face_id in enumerate(face_order):
        row = index // cols
        col = index % cols
        axis = axes[row, col]
        axis.imshow(torch_image_to_numpy(face_dict[face_id]))
        axis.set_title(face_id)
        axis.axis('off')

    plt.tight_layout()
    plt.show()


def show_arrangements(rendered, arrangement_names, face_order, title_prefix=''):
    rows = len(arrangement_names)
    cols = len(face_order)
    fig, axes = plt.subplots(rows, cols, figsize=(2.4 * cols, 2.4 * rows))
    axes = np.asarray(axes, dtype=object).reshape(rows, cols)
    fig.suptitle(title_prefix or 'Rendered arrangements', fontsize=14)

    for row_index, arrangement_name in enumerate(arrangement_names):
        for col_index, face_id in enumerate(face_order):
            axis = axes[row_index, col_index]
            axis.imshow(torch_image_to_numpy(rendered[arrangement_name][face_id]))
            axis.set_title(f'{arrangement_name}:{face_id}', fontsize=9)
            axis.axis('off')

    plt.tight_layout()
    plt.show()


def make_procedural_source_batch(face_order, image_size, grid_size=3, device='cpu'):
    if image_size % grid_size != 0:
        raise ValueError('image_size must be divisible by grid_size')

    palette = PALETTE.to(device)
    tile_size = image_size // grid_size
    y = torch.linspace(0.0, 1.0, image_size, device=device).view(1, image_size, 1)
    x = torch.linspace(0.0, 1.0, image_size, device=device).view(1, 1, image_size)
    batch = []

    for index, _face_id in enumerate(face_order):
        base = palette[index].view(3, 1, 1)
        image = (0.45 * base + 0.35 * x + 0.20 * y).clone()

        top_band = max(2, tile_size // 6)
        image[:, :top_band, :] = torch.tensor([0.08, 0.08, 0.08], device=device).view(3, 1, 1)

        grid_line = max(1, tile_size // 18)
        for step in range(1, grid_size):
            pos = step * tile_size
            image[:, pos - grid_line:pos + grid_line, :] = 0.0
            image[:, :, pos - grid_line:pos + grid_line] = 0.0

        marker = max(3, tile_size // 5)
        marker_color = (1.0 - 0.5 * base)
        for row in range(grid_size):
            for col in range(grid_size):
                top = row * tile_size + max(1, tile_size // 8)
                left = col * tile_size + max(1, tile_size // 8)
                image[:, top:top + marker, left:left + marker] = marker_color

        batch.append(image.clamp(0.0, 1.0))

    return torch.stack(batch, dim=0)


In [ ]:
def rotate_tile_tensor(tile, quarter_turns):
    if quarter_turns not in {0, 1, 2, 3}:
        raise ValueError(f'Unsupported quarter turn count: {quarter_turns}')

    if quarter_turns == 0:
        return tile

    return torch.rot90(tile, k=(-quarter_turns) % 4, dims=(1, 2))


def render_face_grid_torch(grid, source_batch, face_to_index, source_grid_size):
    _batch, _channels, image_height, image_width = source_batch.shape
    tile_height = image_height // source_grid_size
    tile_width = image_width // source_grid_size
    rows = []

    for row in grid:
        tiles = []
        for cell in row:
            face_index = face_to_index[cell['sourceFace']]
            top = cell['sourceRow'] * tile_height
            left = cell['sourceCol'] * tile_width
            tile = source_batch[face_index, :, top:top + tile_height, left:left + tile_width]
            tiles.append(rotate_tile_tensor(tile, cell['rotationQuarterTurns']))
        rows.append(torch.cat(tiles, dim=2))

    return torch.cat(rows, dim=1)


def render_arrangement_torch(arrangement, source_batch, face_to_index, source_grid_size):
    return {
        face_spec['face']: render_face_grid_torch(
            face_spec['grid'],
            source_batch,
            face_to_index,
            source_grid_size,
        )
        for face_spec in arrangement
    }


def render_all_arrangements_torch(spec, source_batch, face_to_index):
    return {
        arrangement_name: render_arrangement_torch(
            arrangement,
            source_batch,
            face_to_index,
            spec['sourceGridSize'],
        )
        for arrangement_name, arrangement in spec['arrangements'].items()
    }


In [ ]:
DEMO_SIZE = 96

demo_source_batch = make_procedural_source_batch(prime_images, DEMO_SIZE, source_grid_size, device='cpu')
demo_source_faces_pil = batch_to_pil_face_dict(demo_source_batch, prime_images)

pil_rendered = render_all_arrangements(spec, demo_source_faces_pil)
torch_rendered = render_all_arrangements_torch(spec, demo_source_batch.to(device), face_to_index)

max_diff = 0.0
for arrangement_name in pil_rendered:
    for face_id, pil_image in pil_rendered[arrangement_name].items():
        pil_tensor = pil_to_tensor(pil_image, device=device)
        diff = (pil_tensor - torch_rendered[arrangement_name][face_id]).abs().max().item()
        max_diff = max(max_diff, diff)

print(f'Max absolute difference between PIL and torch renders: {max_diff:.6f}')
show_face_dict(demo_source_faces_pil, 'Procedural source faces', prime_images)
show_arrangements(clone_rendered_to_cpu(torch_rendered), ['solved', 'scrambled'], prime_images, 'Torch render from procedural faces')


## Toy Targets

We still are **not** asking for a photoreal illusion.

We only pick two tiny goals:

- `solved:F` should look like a circle
- `scrambled:U` should look like diagonal stripes

This is intentionally small. If the optimizer can satisfy even these toy goals, then the end-to-end learning path is working.


In [ ]:
def make_circle_target(image_size, foreground, background, device):
    y, x = torch.meshgrid(
        torch.linspace(-1.0, 1.0, image_size, device=device),
        torch.linspace(-1.0, 1.0, image_size, device=device),
        indexing='ij',
    )
    mask = (torch.sqrt(x**2 + y**2) < 0.55).float().unsqueeze(0)
    fg = torch.tensor(foreground, dtype=torch.float32, device=device).view(3, 1, 1)
    bg = torch.tensor(background, dtype=torch.float32, device=device).view(3, 1, 1)
    return fg * mask + bg * (1.0 - mask)


def make_diagonal_target(image_size, color_a, color_b, device):
    y, x = torch.meshgrid(
        torch.arange(image_size, device=device),
        torch.arange(image_size, device=device),
        indexing='ij',
    )
    stripe_width = max(4, image_size // 12)
    stripes = (((x + y) // stripe_width) % 2).float().unsqueeze(0)
    a = torch.tensor(color_a, dtype=torch.float32, device=device).view(3, 1, 1)
    b = torch.tensor(color_b, dtype=torch.float32, device=device).view(3, 1, 1)
    return a * stripes + b * (1.0 - stripes)


OPT_SIZE = 96
targets = {
    'solved': {
        'F': make_circle_target(
            OPT_SIZE,
            foreground=(0.95, 0.18, 0.22),
            background=(0.98, 0.95, 0.88),
            device=device,
        )
    },
    'scrambled': {
        'U': make_diagonal_target(
            OPT_SIZE,
            color_a=(0.10, 0.35, 0.95),
            color_b=(0.90, 0.95, 1.00),
            device=device,
        )
    },
}

print('Targeted faces:', {name: list(face_targets.keys()) for name, face_targets in targets.items()})
show_face_dict(
    {
        'solved:F target': targets['solved']['F'],
        'scrambled:U target': targets['scrambled']['U'],
    },
    'Toy targets',
    ['solved:F target', 'scrambled:U target'],
)


In [ ]:
def compute_target_loss(rendered, targets):
    total = torch.tensor(0.0, device=device)
    components = {}

    for arrangement_name, target_faces in targets.items():
        for face_id, target in target_faces.items():
            mse = (rendered[arrangement_name][face_id] - target).pow(2).mean()
            components[f'{arrangement_name}:{face_id}'] = mse
            total = total + mse

    return total, components


def total_variation_loss(batch):
    horizontal = (batch[:, :, :, 1:] - batch[:, :, :, :-1]).abs().mean()
    vertical = (batch[:, :, 1:, :] - batch[:, :, :-1, :]).abs().mean()
    return horizontal + vertical


def render_from_logits(source_logits):
    source_batch = torch.sigmoid(source_logits)
    rendered = render_all_arrangements_torch(spec, source_batch, face_to_index)
    return source_batch, rendered


torch.manual_seed(7)
source_logits = torch.nn.Parameter(torch.randn(len(prime_images), 3, OPT_SIZE, OPT_SIZE, device=device) * 0.15)

initial_source_batch = torch.sigmoid(source_logits).detach().cpu()
show_face_dict(batch_to_pil_face_dict(initial_source_batch, prime_images), 'Initial learnable source faces', prime_images)


In [ ]:
optimizer = torch.optim.Adam([source_logits], lr=0.08)
tv_weight = 0.002
num_steps = 250
snapshot_steps = {0, 25, 100, 250}

loss_history = []
component_history = []
snapshots = {}

for step in range(num_steps + 1):
    optimizer.zero_grad()
    source_batch, rendered = render_from_logits(source_logits)
    reconstruction_loss, components = compute_target_loss(rendered, targets)
    tv_loss = total_variation_loss(source_batch)
    total_loss = reconstruction_loss + tv_weight * tv_loss

    loss_history.append(float(total_loss.detach().cpu()))
    component_history.append({name: float(value.detach().cpu()) for name, value in components.items()})

    if step in snapshot_steps:
        snapshots[step] = {
            'source_batch': source_batch.detach().cpu().clone(),
            'rendered': clone_rendered_to_cpu(rendered),
            'total_loss': float(total_loss.detach().cpu()),
            'components': {name: float(value.detach().cpu()) for name, value in components.items()},
        }
        print(
            f'step={step:>3} '
            f'total_loss={snapshots[step]["total_loss"]:.5f} '
            f'components={snapshots[step]["components"]}'
        )

    if step == num_steps:
        break

    total_loss.backward()
    optimizer.step()


In [ ]:
final_source_batch, final_rendered = render_from_logits(source_logits)
final_source_batch = final_source_batch.detach().cpu()
final_rendered_cpu = clone_rendered_to_cpu(final_rendered)

fig, axis = plt.subplots(1, 1, figsize=(7, 3.5))
axis.plot(loss_history, label='total loss')
axis.set_title('Optimization loss curve')
axis.set_xlabel('Step')
axis.set_ylabel('Loss')
axis.grid(alpha=0.3)
axis.legend()
plt.show()

show_face_dict(batch_to_pil_face_dict(final_source_batch, prime_images), 'Optimized source faces', prime_images)
show_arrangements(final_rendered_cpu, ['solved', 'scrambled'], prime_images, 'Final torch renders')
show_face_dict(
    {
        'target solved:F': targets['solved']['F'].detach().cpu(),
        'rendered solved:F': final_rendered_cpu['solved']['F'],
        'target scrambled:U': targets['scrambled']['U'].detach().cpu(),
        'rendered scrambled:U': final_rendered_cpu['scrambled']['U'],
    },
    'Target vs rendered comparison',
    ['target solved:F', 'rendered solved:F', 'target scrambled:U', 'rendered scrambled:U'],
)


In [ ]:
OUTPUT_ROOT = REPO_DIR / 'output' / 'colab-optimization-sandbox'
SOURCE_OUTPUT_DIR = OUTPUT_ROOT / 'source-faces'
RENDER_OUTPUT_DIR = OUTPUT_ROOT / 'rendered'

SOURCE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

final_source_faces_pil = batch_to_pil_face_dict(final_source_batch, prime_images)
for face_id, image in final_source_faces_pil.items():
    image.save(SOURCE_OUTPUT_DIR / f'{face_id}.png')

save_rendered_faces(
    {face_id: tensor_to_pil(image) for face_id, image in final_rendered_cpu['solved'].items()},
    RENDER_OUTPUT_DIR / 'solved',
)
save_rendered_faces(
    {face_id: tensor_to_pil(image) for face_id, image in final_rendered_cpu['scrambled'].items()},
    RENDER_OUTPUT_DIR / 'scrambled',
)

print('Saved source faces to:', SOURCE_OUTPUT_DIR)
print('Saved rendered solved faces to:', RENDER_OUTPUT_DIR / 'solved')
print('Saved rendered scrambled faces to:', RENDER_OUTPUT_DIR / 'scrambled')
